# Targeted sample review

Use the existing Codex desktop windows for semantic review. This notebook stages
versioned local assignments, verifies their receipts, and tracks parent dispositions
by sample and origin. It retains all original text and holds. A completed sample
supports a separate decision on whether to expand the nonparagraph review scope.

## Bind the frozen sample

Read the verified 08 candidate, reference plan and stratified 4,635-parent roster.

In [ ]:
import json
import tempfile
from pathlib import Path

import nbformat
import pandas as pd
from IPython.display import display
from pyspark.sql import Row
from pyspark.sql import functions as F

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "notebooks/08_targeted_chunk_review.ipynb").is_file()
)
notebook_08 = nbformat.read(ROOT / "notebooks/08_targeted_chunk_review.ipynb", as_version=4)
parent_cells = {cell.id: cell.source for cell in notebook_08.cells if cell.cell_type == "code"}
source = {"__name__": "__main__"}
exec(compile(parent_cells["08-bind"], "08_targeted_chunk_review.ipynb:08-bind", "exec"), source)
parents = source["parents"]
plans = source["plans"]
spark = source["spark"]
sha256_file = source["sha256_file"]
canonical_sha256 = source["canonical_sha256"]
make_review_packets = source["make_review_packets"]
policy_dir = source["output"]
policy_manifest_path = policy_dir / "MANIFEST.json"
policy_manifest = json.loads(policy_manifest_path.read_text(encoding="utf-8"))
policy_sha = sha256_file(policy_manifest_path)
policy_path = policy_dir / "CHUNK_POLICY_V1_CANDIDATE.json"
if sha256_file(policy_path) != policy_manifest["policy_sha256"]:
    raise ValueError("The candidate policy receipt failed verification.")
policy = json.loads(policy_path.read_text(encoding="utf-8"))
if policy["reference_manifest_sha256"] != source["review_policy"]["reference_manifest_sha256"]:
    raise ValueError("The sample belongs to another reference plan.")
sample = source["upstream"]["bound_review_table"](
    policy_dir, "semantic_sample", policy_manifest["tables"]["semantic_sample"]
)


## Publish resumable whole-parent assignments

Register the prior replays when their parent and receipt hashes match. Keep the diagnostic and first sample batch origins visible. Assign all other parents to disjoint batches. Drop no original chunks or split memberships.

In [ ]:
from collections import Counter

CACHE = ROOT / ".cache/chunk-policy-review"
prior_dirs = {
    "diagnostic": (CACHE / "96e814e5c00fefa5a9ef", "TRIAL_REPLAY.json"),
    "sample-001": (CACHE / "3d89641f2a281193dc7a", "BATCH_REPLAY.json"),
}
registered = {}
ancestors = {}
for label, (directory, name) in prior_dirs.items():
    path = directory / name
    replay = json.loads(path.read_text(encoding="utf-8"))
    if replay["policy_manifest_sha256"] != policy_sha or replay[
        "trial_manifest_sha256"
    ] != sha256_file(directory / "MANIFEST.json"):
        raise ValueError(f"The replay for {label} does not match the frozen sample.")
    for batch, digest in replay["receipts"].items():
        if sha256_file(directory / "decisions" / f"{batch}.jsonl") != digest:
            raise ValueError(f"The registered receipt for {label}/{batch} changed.")
    for parent in replay["parents"]:
        release_row = parent["release_row"]
        if release_row in registered:
            raise ValueError("Two historical replays claim the same sampled parent.")
        registered[release_row] = {**parent, "review_origin": label}
    ancestors[label] = sha256_file(path)
sample_rows = {
    row.release_row: row
    for row in sample.select("release_row", "clean_sha256", "sample", "risk", "ml_split").collect()
}
if len(sample_rows) != 4635 or any(
    key not in sample_rows
    or any(
        row[field] != sample_rows[key][field]
        for field in ("clean_sha256", "sample", "risk", "ml_split")
        if field in row
    )
    or row["body_sha256"] != sample_rows[key].clean_sha256
    for key, row in registered.items()
):
    raise ValueError("The review ledger does not match the frozen sample membership.")

core = source["CODEPOINT_WINDOW"]
margin = source["CONTEXT_WIDTH"]
cap = 48
run_identity = {
    "policy_manifest_sha256": policy_sha,
    "reference_manifest_sha256": policy["reference_manifest_sha256"],
    "registered_replay_sha256": ancestors,
    "batch_windows_target": cap,
    "policy": "full_targeted_sample_v1",
}
RUN = CACHE / "full-sample-v1"
if not RUN.exists():
    pending = sample.where(~F.col("release_row").isin(*sorted(registered)))
    review_bodies = (
        pending.join(parents.select("release_row", "clean_text", "paragraphs"), "release_row")
        .join(plans.select("release_row", "plan"), "release_row")
        .orderBy("ml_split", "draw_key", "release_row")
    )
    with tempfile.TemporaryDirectory(prefix=".sample-run-", dir=CACHE) as temp:
        stage = Path(temp) / "snapshot"
        stage.mkdir(mode=0o700)
        (stage / "decisions").mkdir(mode=0o700)
        (stage / "dispatch").mkdir(mode=0o700)
        line = 0
        batch_items = []
        batches = {}
        roster = set()
        window_counts = Counter()
        packet_file = stage / "PACKETS.jsonl"

        def save_items(items):
            batch = f"sample-{len(batches) + 2:04d}"
            file = stage / f"{batch}.json"
            document = {
                "batch": batch,
                "units": items,
                "receipt": f"decisions/{batch}.jsonl",
            }
            file.write_text(json.dumps(document, indent=2) + "\n", encoding="utf-8")
            batches[batch] = {
                "file": file.name,
                "sha256": sha256_file(file),
                "windows": len(items),
                "parents": len({item["release_row"] for item in items}),
            }
            file.chmod(0o400)

        with packet_file.open("x", encoding="utf-8") as handle:
            for row in review_bodies.toLocalIterator():
                if row.release_row in roster:
                    raise ValueError("A parent appears twice in the pending sample.")
                roster.add(row.release_row)
                parent = Row(
                    release_row=row.release_row,
                    clean_sha256=row.clean_sha256,
                    clean_chars=row.clean_chars,
                    plan=row.plan,
                    plan_sha256=canonical_sha256(
                        {
                            "reference_manifest_sha256": policy["reference_manifest_sha256"],
                            "body_sha256": row.clean_sha256,
                            "ends": row.plan.ends,
                        }
                    ),
                    review_units=[
                        Row(
                            unit_ordinal=ordinal,
                            start=start,
                            end=min(start + core, row.clean_chars),
                            context_start=max(0, start - margin),
                            context_end=min(start + core + margin, row.clean_chars),
                        )
                        for ordinal, start in enumerate(range(0, row.clean_chars, core))
                    ],
                )
                targets = (
                    row.nonparagraph_offsets if row.sample == "nonparagraph" else row.plan.ends[:-1]
                )
                selected_ordinals = {end // core for end in targets} if targets else {0}
                selected = [
                    packet
                    for packet in make_review_packets(parent, row.clean_text, row.paragraphs)
                    if packet["unit_ordinal"] in selected_ordinals
                ]
                if len(selected) != len(selected_ordinals):
                    raise ValueError("A sample parent lost an active review window.")
                if batch_items and len(batch_items) + len(selected) > cap:
                    save_items(batch_items)
                    batch_items = []
                for packet in selected:
                    handle.write(json.dumps(packet, ensure_ascii=False) + "\n")
                    line += 1
                    batch_items.append(
                        {
                            "line": line,
                            "release_row": row.release_row,
                            "sample": row.sample,
                            "risk": row.risk,
                            "format": row.format,
                            "packet_sha256": packet["packet_sha256"],
                            "target_boundary_ids": [
                                end for end in targets if packet["start"] <= end < packet["end"]
                            ],
                        }
                    )
                window_counts[row.sample] += len(selected)
        if batch_items:
            save_items(batch_items)
        if roster != set(sample_rows) - set(registered):
            raise ValueError("The new batch roster does not cover the remaining frozen sample.")
        packet_file.chmod(0o400)
        manifest = {
            **run_identity,
            "packet_sha256": sha256_file(packet_file),
            "registered_parents": len(registered),
            "new_parents": len(roster),
            "new_windows": line,
            "windows_by_sample": dict(window_counts),
            "batches": batches,
        }
        manifest_path = stage / "MANIFEST.json"
        manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
        manifest_path.chmod(0o400)
        if RUN.exists():
            raise FileExistsError("A full sample run appeared while this one was staged.")
        stage.rename(RUN)

run_manifest = json.loads((RUN / "MANIFEST.json").read_text(encoding="utf-8"))
if any(run_manifest.get(k) != v for k, v in run_identity.items()):
    raise ValueError("The local run belongs to another frozen sample.")
if sha256_file(RUN / "PACKETS.jsonl") != run_manifest["packet_sha256"]:
    raise ValueError("The shared packet file failed verification.")
assignments = {}
seen_parents, seen_lines = set(registered), set()
for label, entry in run_manifest["batches"].items():
    path = RUN / entry["file"]
    if sha256_file(path) != entry["sha256"]:
        raise ValueError(f"The assignment {label} changed.")
    assignment = json.loads(path.read_text(encoding="utf-8"))
    assigned = {item["release_row"] for item in assignment["units"]}
    lines = {item["line"] for item in assignment["units"]}
    if (
        assignment["batch"] != label
        or assignment["receipt"] != f"decisions/{label}.jsonl"
        or len(lines) != entry["windows"]
        or len(lines) != len(assignment["units"])
        or len(assigned) != entry["parents"]
        or assigned & seen_parents
        or lines & seen_lines
    ):
        raise ValueError(f"Overlapping or incorrect sample assignment for {label}.")
    assignments[label] = assignment
    seen_parents.update(assigned)
    seen_lines.update(lines)
if seen_parents != set(sample_rows) or seen_lines != set(range(1, run_manifest["new_windows"] + 1)):
    raise ValueError("The complete assignment roster is inconsistent.")
print(
    f"Registered {len(registered)} previously replayed parents and prepared "
    f"{run_manifest['new_windows']} windows in {len(assignments)} assignable batches "
    f"covering {run_manifest['new_parents']} additional parents."
)
print(f"Full targeted sample run: {RUN.relative_to(ROOT)}")


## Bind the three resumable reviewer lanes

Each window owns one lane and processes its assignments in order. Finished receipts and checkpoints allow the same lane to resume without re-reviewing completed parents.

In [ ]:
lanes_dir = RUN / "dispatch"
batch_labels = sorted(assignments)
lane_summary = []
covered_batches = set()
for slot in range(3):
    label = f"reviewer-lane-{slot + 1:03d}"
    queue = batch_labels[slot::3]
    lane = {
        "lane": label,
        "run_manifest_sha256": sha256_file(RUN / "MANIFEST.json"),
        "packet_sha256": run_manifest["packet_sha256"],
        "assignments": [{"batch": batch, **run_manifest["batches"][batch]} for batch in queue],
    }
    path = lanes_dir / f"{label}.json"
    if path.exists():
        if json.loads(path.read_text(encoding="utf-8")) != lane:
            raise ValueError(f"The published reviewer lane changed: {label}.")
    else:
        with path.open("x", encoding="utf-8") as handle:
            handle.write(json.dumps(lane, indent=2) + "\n")
        path.chmod(0o400)
    if covered_batches.intersection(queue):
        raise ValueError("Two reviewer lanes claim the same batch.")
    covered_batches.update(queue)
    lane_summary.append(
        {
            "Lane": label,
            "Batches": len(queue),
            "Windows": sum(run_manifest["batches"][batch]["windows"] for batch in queue),
        }
    )
if covered_batches != set(assignments):
    raise ValueError("The lanes do not cover the complete batch plan.")
display(pd.DataFrame(lane_summary))


## Replay completed batches

Verify packet membership, preserve undispatched boundaries and recalculate the reference token counts. An escalation stays open for case-level adjudication.

In [ ]:
import tiktoken

encoder = tiktoken.get_encoding(policy["reference_encoding"])
budget = policy["reference_content_tokens"]
replay_dir = RUN / "replays"
replay_dir.mkdir(mode=0o700, exist_ok=True)
reviewed = dict(registered)
replay_counts = Counter()
pending_replay = []

for batch, assignment in assignments.items():
    receipt = RUN / assignment["receipt"]
    replay_file = replay_dir / f"{batch}.json"
    if not receipt.is_file():
        if replay_file.is_file():
            raise ValueError(f"The checked receipt disappeared for {batch}.")
        continue
    receipt_sha = sha256_file(receipt)
    assignment_sha = run_manifest["batches"][batch]["sha256"]
    if replay_file.is_file():
        saved = json.loads(replay_file.read_text(encoding="utf-8"))
        if (saved["receipt_sha256"], saved["assignment_sha256"]) != (receipt_sha, assignment_sha):
            raise ValueError(f"The checked inputs changed for {batch}.")
        for row in saved["parents"]:
            if row["release_row"] in reviewed:
                raise ValueError("Two replays contain the same sampled parent.")
            reviewed[row["release_row"]] = row
        replay_counts.update(saved["decision_counts"])
    else:
        pending_replay.append((batch, receipt, receipt_sha, assignment_sha))

if pending_replay:
    assignments_to_replay = {
        batch: {unit["packet_sha256"]: unit for unit in assignments[batch]["units"]}
        for batch, _, _, _ in pending_replay
    }
    target_lines = {
        unit["line"]: digest
        for units in assignments_to_replay.values()
        for digest, unit in units.items()
    }
    packets = {}
    with (RUN / "PACKETS.jsonl").open(encoding="utf-8") as handle:
        for line, raw in enumerate(handle, start=1):
            if line not in target_lines:
                continue
            packet = json.loads(raw)
            digest = packet.pop("packet_sha256")
            if digest != target_lines[line] or canonical_sha256(packet) != digest:
                raise ValueError("A frozen packet line does not match its assignment.")
            packets[digest] = packet
    if set(packets) != set(target_lines.values()):
        raise ValueError("The selected packet lines could not be replayed.")

    ids = sorted({packet["release_row"] for packet in packets.values()})
    corpus = {
        row.release_row: row
        for row in parents.where(F.col("release_row").isin(*ids))
        .select("release_row", "clean_text")
        .collect()
    }
    reference = {
        row.release_row: row
        for row in plans.where(F.col("release_row").isin(*ids))
        .select("release_row", "clean_sha256", "plan")
        .collect()
    }
    if set(corpus) != set(ids) or set(reference) != set(ids):
        raise ValueError("The full reference parents could not be rebound.")

    for batch, receipt, receipt_sha, assignment_sha in pending_replay:
        expected = assignments_to_replay[batch]
        entries = [json.loads(line) for line in receipt.read_text(encoding="utf-8").splitlines()]
        if len(entries) != len(expected) or {row["packet_sha256"] for row in entries} != set(
            expected
        ):
            raise ValueError(f"The receipt roster is incomplete or duplicated: {batch}.")
        parent_decisions = {}
        counts = Counter()
        for entry in entries:
            digest = entry["packet_sha256"]
            packet = packets[digest]
            assigned = expected[digest]
            chosen = entry["selected_boundary_ids"]
            status = entry["status"]
            evidence = entry["evidence_outcome"]
            baseline = packet["baseline_boundary_ids"]
            if (
                not isinstance(chosen, list)
                or any(type(position) is not int for position in chosen)
                or chosen != sorted(set(chosen))
                or not set(chosen) <= set(packet["candidate_boundary_ids"])
                or any(
                    not isinstance(entry.get(key), str) or not entry[key].strip()
                    for key in ("reason", "reviewer", "reviewer_version")
                )
            ):
                raise ValueError(f"Invalid selected boundaries or provenance in {batch}.")
            if not (
                (
                    status == "accept"
                    and chosen == baseline
                    and evidence in {"none", "boundary_preference"}
                )
                or (
                    status == "adjust"
                    and chosen != baseline
                    and evidence == "material_evidence_break"
                    and set(baseline) - set(chosen) <= set(assigned["target_boundary_ids"])
                )
                or (
                    status == "escalate"
                    and not chosen
                    and evidence in {"material_evidence_break", "uncertain"}
                )
            ):
                raise ValueError(f"The evidence finding disagrees with the decision in {batch}.")
            parent_decisions.setdefault(packet["release_row"], []).append((entry, packet))
            counts[status] += 1
        rows = []
        for parent_id, decisions in sorted(parent_decisions.items()):
            if parent_id in reviewed:
                raise ValueError("Two sample receipts claim the same parent.")
            meta = sample_rows[parent_id]
            ref = reference[parent_id]
            body = corpus[parent_id].clean_text
            baseline = ref.plan.ends
            plan_sha = canonical_sha256(
                {
                    "reference_manifest_sha256": policy["reference_manifest_sha256"],
                    "body_sha256": ref.clean_sha256,
                    "ends": baseline,
                }
            )
            if not all(
                packet["body_sha256"] == meta.clean_sha256 == ref.clean_sha256
                and packet["plan_sha256"] == plan_sha
                and packet["context_text"] == body[packet["context_start"] : packet["context_end"]]
                for _, packet in decisions
            ):
                raise ValueError("A review packet did not match its reference text and plan.")
            ends = set(baseline)
            parent_status = "boundary_review_complete"
            token_counts = None
            if any(entry["status"] == "escalate" for entry, _ in decisions):
                parent_status = "escalated"
            else:
                for entry, packet in decisions:
                    ends.difference_update(packet["baseline_boundary_ids"])
                    ends.update(entry["selected_boundary_ids"])
                ends = sorted(ends)
                if ends[-1:] != [len(body)] or ends[0] <= 0:
                    raise ValueError("A completed parent failed coverage.")
                token_counts = [
                    len(encoder.encode_ordinary(body[start:end]))
                    for start, end in zip([0, *ends[:-1]], ends, strict=True)
                ]
                if any(count < 1 or count > budget for count in token_counts):
                    parent_status = "reference_budget_review_required"
            row = {
                "release_row": parent_id,
                "sample": meta.sample,
                "risk": meta.risk,
                "ml_split": meta.ml_split,
                "body_sha256": meta.clean_sha256,
                "review_origin": batch,
                "status": parent_status,
                "internal_end_offsets": ends[:-1]
                if parent_status == "boundary_review_complete"
                else None,
                "token_counts": token_counts
                if parent_status == "boundary_review_complete"
                else None,
                "evidence_outcomes": sorted({entry["evidence_outcome"] for entry, _ in decisions}),
            }
            rows.append(row)
            reviewed[parent_id] = row
        if sha256_file(receipt) != receipt_sha:
            raise ValueError(f"The receipt changed during replay: {batch}.")
        record = {
            "batch": batch,
            "receipt_sha256": receipt_sha,
            "assignment_sha256": assignment_sha,
            "decision_counts": dict(counts),
            "parents": rows,
        }
        with tempfile.NamedTemporaryFile(
            dir=replay_dir, mode="w", encoding="utf-8", delete=False
        ) as handle:
            handle.write(json.dumps(record, indent=2) + "\n")
            staged = Path(handle.name)
        staged.rename(replay_dir / f"{batch}.json")
        replay_counts.update(counts)

print(f"Replayed {len(reviewed)} of {len(sample_rows)} frozen sampled parents.")
display(pd.DataFrame(reviewed.values()).groupby(["sample", "status"]).size().rename("parents"))


## Adjudicate the five first-batch escalations

Preserve the original reviews. A verified repair or an explicit hold gives the parent a recorded disposition. A failed replay remains open.

In [ ]:
a_dir = prior_dirs["sample-001"][0] / "adjudication"
a_manifest = json.loads((a_dir / "MANIFEST.json").read_text(encoding="utf-8"))
required = set(a_manifest["parents"])
case_references = {
    row.release_row: row
    for row in plans.where(F.col("release_row").isin(*sorted(required)))
    .select("release_row", "clean_sha256", "plan")
    .collect()
}
case_text = {
    row.release_row: row
    for row in parents.where(F.col("release_row").isin(*sorted(required)))
    .select("release_row", "clean_text", "paragraphs")
    .collect()
}
adjudications = {}
sentence_break = source["upstream"]["SENTENCE_BREAK"]
whitespace_break = source["upstream"]["WHITESPACE_BREAK"]
for name, digest in a_manifest["sha256_by_assignment"].items():
    assignment_path = a_dir / name
    if sha256_file(assignment_path) != digest:
        raise ValueError(f"The adjudication assignment changed: {name}.")
    assignment = json.loads(assignment_path.read_text(encoding="utf-8"))
    if assignment["sample_batch_replay_sha256"] != ancestors["sample-001"]:
        raise ValueError("Adjudication is bound to another first-batch replay.")
    receipt_path = a_dir / assignment["receipt"]
    if not receipt_path.is_file():
        continue
    cases = {row["release_row"] for row in assignment["cases"]}
    entries = [json.loads(line) for line in receipt_path.read_text(encoding="utf-8").splitlines()]
    if len(entries) != len(cases) or {entry["release_row"] for entry in entries} != cases:
        raise ValueError(f"Incorrect adjudication receipt roster: {name}.")
    for entry in entries:
        key = entry["release_row"]
        ref = case_references[key]
        record = case_text[key]
        disposition = entry["disposition"]
        if (
            key in adjudications
            or registered[key]["status"] != "escalated"
            or entry["body_sha256"] != ref.clean_sha256
            or disposition not in {"retain_reference", "repair_reference", "hold_for_model_input"}
            or any(
                not isinstance(entry.get(field), str) or not entry[field].strip()
                for field in ("reason", "reviewer", "reviewer_version")
            )
        ):
            raise ValueError("Invalid adjudication identity or evidence.")
        ends = entry["replacement_ends"]
        if disposition == "repair_reference":
            plausible = set(ref.plan.ends)
            plausible.update(span.start for span in record.paragraphs)
            plausible.update(match.end() for match in sentence_break.finditer(record.clean_text))
            plausible.update(match.end() for match in whitespace_break.finditer(record.clean_text))
            if not (
                isinstance(ends, list)
                and ends
                and all(type(end) is int for end in ends)
                and ends == sorted(set(ends))
                and ends[0] > 0
                and ends[-1] == len(record.clean_text)
                and set(ends) <= plausible
                and ends != ref.plan.ends
            ):
                raise ValueError("Adjudicated repair violates the parent offset policy.")
        elif ends is not None:
            raise ValueError("Only a repair may provide replacement boundaries.")
        if disposition == "retain_reference":
            ends = ref.plan.ends
        counts = None
        if ends is not None:
            counts = [
                len(encoder.encode_ordinary(record.clean_text[start:end]))
                for start, end in zip([0, *ends[:-1]], ends, strict=True)
            ]
            if any(count < 1 or count > budget for count in counts):
                disposition = "reference_budget_review_required"
        adjudications[key] = {
            **entry,
            "verified_disposition": disposition,
            "receipt_sha256": sha256_file(receipt_path),
            "reference_token_counts": counts,
        }
print(f"Replayed {len(adjudications)} of {len(required)} first-batch adjudications.")
if adjudications:
    statuses = Counter(row["verified_disposition"] for row in adjudications.values())
    display(pd.DataFrame(statuses.items(), columns=["Verified disposition", "Parents"]))
    audit = {
        "first_batch_replay_sha256": ancestors["sample-001"],
        "required_parents": sorted(required),
        "adjudications": [adjudications[key] for key in sorted(adjudications)],
    }
    path = a_dir / "ADJUDICATION_REPLAY.json"
    with tempfile.NamedTemporaryFile(dir=a_dir, mode="w", encoding="utf-8", delete=False) as handle:
        handle.write(json.dumps(audit, indent=2) + "\n")
        staged = Path(handle.name)
    staged.replace(path)


effective = {key: value["status"] for key, value in reviewed.items()}
for key, decision in adjudications.items():
    disposition = decision["verified_disposition"]
    effective[key] = (
        "boundary_review_complete"
        if disposition in {"retain_reference", "repair_reference"}
        else disposition
    )
effective_counts = Counter(effective.values())
batch_receipts = sum((RUN / a["receipt"]).is_file() for a in assignments.values())
progress = {
    "run_manifest_sha256": sha256_file(RUN / "MANIFEST.json"),
    "sample_parents": len(sample_rows),
    "replayed_parents": len(effective),
    "pending_first_review_parents": len(sample_rows) - len(effective),
    "current_dispositions": dict(effective_counts),
    "verified_first_batch_adjudications": len(adjudications),
    "batches_with_final_receipts": batch_receipts,
    "batches_in_run": len(assignments),
}
with tempfile.NamedTemporaryFile(dir=RUN, mode="w", encoding="utf-8", delete=False) as handle:
    handle.write(json.dumps(progress, indent=2) + "\n")
    staged = Path(handle.name)
staged.replace(RUN / "PROGRESS.json")
display(pd.DataFrame(effective_counts.items(), columns=["Current disposition", "Parents"]))


## Close the frozen sample and report stratified findings

Publish a closure only when every sampled parent has a verified replay. Keep first-pass material findings separate from final open dispositions.

In [ ]:
if len(reviewed) == len(sample_rows):
    design_rows = sample.select(
        "release_row",
        "sample",
        "risk",
        "format",
        "length_band",
        "ml_split",
        "stratum_parents",
        "stratum_sample",
    ).collect()
    if {row.release_row for row in design_rows} != set(reviewed):
        raise ValueError("A final sample replay is outside the frozen cohort.")
    summary = {}
    for row in design_rows:
        parent = reviewed[row.release_row]
        weight = row.stratum_parents / row.stratum_sample
        result = summary.setdefault(
            row.sample,
            {
                "parents": 0,
                "material_parents": 0,
                "unresolved_parents": 0,
                "weighted_total": 0.0,
                "weighted_material": 0.0,
                "weighted_unresolved": 0.0,
            },
        )
        material = "material_evidence_break" in parent["evidence_outcomes"]
        unresolved = effective[row.release_row] in {"escalated", "reference_budget_review_required"}
        result["parents"] += 1
        result["material_parents"] += int(material)
        result["unresolved_parents"] += int(unresolved)
        result["weighted_total"] += weight
        result["weighted_material"] += weight * material
        result["weighted_unresolved"] += weight * unresolved
    for row in summary.values():
        row["weighted_material_share"] = row["weighted_material"] / row["weighted_total"]
        row["weighted_unresolved_share"] = row["weighted_unresolved"] / row["weighted_total"]
    evidence = {
        **progress,
        "status": "sample_replay_complete",
        "measure": "first-pass reviewer findings; weighted by the frozen stratum inclusion rate",
        "stratified_summary": summary,
    }
    target = RUN / "SAMPLE_CLOSURE.json"
    with tempfile.NamedTemporaryFile(dir=RUN, mode="w", encoding="utf-8", delete=False) as handle:
        handle.write(json.dumps(evidence, indent=2) + "\n")
        staged = Path(handle.name)
    staged.replace(target)
    display(pd.DataFrame.from_dict(summary, orient="index"))
else:
    print(f"Full sample closure is pending {len(sample_rows) - len(reviewed)} parent replays.")


## Recheck exact leakage with the sample review offsets

Apply the verified sample offsets as a separate candidate. Preserve reference holds whose source slices remain and add all exact cross-split holds on the new candidate.

In [ ]:
if len(reviewed) == len(sample_rows):
    overrides = {
        key: row["internal_end_offsets"]
        for key, row in reviewed.items()
        if row["status"] == "boundary_review_complete"
    }
    for key, row in adjudications.items():
        if row["verified_disposition"] == "repair_reference":
            overrides[key] = row["replacement_ends"][:-1]
    parent_ends = spark.createDataFrame(
        list(overrides.items()), "release_row long, reviewed_internal_ends array<int>"
    )
    proposed = (
        plans.join(F.broadcast(parent_ends), "release_row", "left")
        .withColumn(
            "candidate_ends",
            F.when(
                F.col("reviewed_internal_ends").isNotNull(),
                F.concat("reviewed_internal_ends", F.array("clean_chars")),
            ).otherwise(F.col("plan.ends")),
        )
        .select(
            "release_row",
            "clean_sha256",
            "ml_split",
            "candidate_ends",
            F.posexplode("candidate_ends").alias("ordinal", "end"),
        )
        .withColumn("start", F.coalesce(F.get("candidate_ends", F.col("ordinal") - 1), F.lit(0)))
        .drop("candidate_ends")
        .join(
            parents.select("release_row", "clean_sha256", "clean_text"),
            ["release_row", "clean_sha256"],
        )
        .withColumn("codepoints", F.col("end") - F.col("start"))
        .withColumn(
            "chunk_sha256", F.sha2(F.substring("clean_text", F.col("start") + 1, "codepoints"), 256)
        )
        .drop("clean_text")
        .withColumn(
            "split_mask",
            F.when(F.col("ml_split") == "train", 1)
            .when(F.col("ml_split") == "validation", 2)
            .otherwise(4),
        )
        .persist()
    )
    exact = (
        proposed.groupBy("chunk_sha256", "codepoints")
        .agg(F.bit_or("split_mask").alias("split_mask"))
        .where(F.col("split_mask").isin(3, 5, 6, 7))
        .select("chunk_sha256", "codepoints")
        .persist()
    )
    reference_holds = source["upstream"]["bound_review_table"](
        policy_dir, "exact_chunk_holds", policy_manifest["tables"]["exact_chunk_holds"]
    ).select("release_row", "start", "end", "chunk_sha256")
    retained_keys = proposed.join(
        F.broadcast(reference_holds), ["release_row", "start", "end", "chunk_sha256"]
    ).select("release_row", "ordinal")
    refreshed_keys = proposed.join(F.broadcast(exact), ["chunk_sha256", "codepoints"]).select(
        "release_row", "ordinal"
    )
    all_keys = retained_keys.unionByName(refreshed_keys).distinct()
    post_hold = (
        proposed.join(F.broadcast(all_keys), ["release_row", "ordinal"], "left_anti")
        .groupBy("chunk_sha256", "codepoints")
        .agg(F.bit_or("split_mask").alias("split_mask"))
        .where(F.col("split_mask").isin(3, 5, 6, 7))
        .count()
    )
    if post_hold:
        raise ValueError("Exact cross-split reuse survived the candidate hold recheck.")
    audit = {
        "status": "sample_candidate_exact_recheck_complete",
        "sample_closure_sha256": sha256_file(RUN / "SAMPLE_CLOSURE.json"),
        "reviewed_parent_plans_used": len(overrides),
        "candidate_chunks": proposed.count(),
        "candidate_cross_split_exact_groups": exact.count(),
        "held_occurrences": all_keys.count(),
        "exact_cross_split_groups_after_holds": post_hold,
    }
    with tempfile.NamedTemporaryFile(dir=RUN, mode="w", encoding="utf-8", delete=False) as handle:
        handle.write(json.dumps(audit, indent=2) + "\n")
        staged = Path(handle.name)
    staged.replace(RUN / "SAMPLE_CANDIDATE_EXACT_RECHECK.json")
    print(audit)
    proposed.unpersist()
    exact.unpersist()
